# 03 · Evaluación de los modelos en validación

Calcula las métricas de un modelo entrenado sobre la partición de validación y
analiza sus errores. Con él se hizo la comparación inicial de YOLOv8n,
YOLOv8s, YOLO26n y YOLO26s.

## Guía de lectura y ejecución

| Aspecto | Descripción |
|---|---|
| **Papel en el proyecto** | Calcula las métricas de cada modelo entrenado sobre validación y analiza sus errores. |
| **Cuándo abrirlo** | Para ver la comparación inicial de modelos (Tablas 1 y 2 de la memoria). |
| **Comportamiento por defecto** | No evalúa (`RUN_STANDARD_EVALUATION=False`, `RUN_ERROR_ANALYSIS=False`): muestra las evaluaciones guardadas del YOLO26s inicial y la comparación de los cuatro modelos. |
| **Entradas principales** | Modelos entrenados en `artifacts/experiments/`. |
| **Salidas principales** | Métricas y análisis de errores dentro de la carpeta de cada experimento. |
| **Continuación** | `05_DFire_barrido_umbrales.ipynb`. |

> **Para ejecutarlo:** usar el entorno Docker de notebooks (sección 4 del
> `README.md` principal). Volver a ejecutarlo requiere resultados intermedios
> (`artifacts/experiments/` y otras carpetas de `artifacts/`) que no están en el
> repositorio; ver `notebooks/README.md`.

## 1. Parámetros

- `MODEL_KEY` o `EXPERIMENT_ID`: modelo que se evalúa. Por defecto, el YOLO26s
  de la comparación inicial, entrenado y evaluado a 640 px.
- `RUN_STANDARD_EVALUATION=True`: calcula las métricas estándar de Ultralytics
  (precisión, recall y mAP).
- `RUN_ERROR_ANALYSIS=True`: cuenta aciertos, falsos positivos, falsos
  negativos e imágenes sin humo ni fuego con alarma, con un umbral de confianza
  fijo (`OPERATING_CONFIDENCE`, 0,25).

Por defecto trabaja con validación. El test se reserva para la evaluación final
del notebook 11.

In [1]:
RUN_STANDARD_EVALUATION = False
RUN_ERROR_ANALYSIS = False
EVALUATION_SPLIT = "val"       # val para comparar/decidir; test solo al final
MODEL_KEY = "yolo26s"
EXPERIMENT_ID = "yolo26s_dfire_seed42_20260830T225658Z"  # YOLO26s de la comparación inicial (640 px).
# Con EXPERIMENT_ID = None se usa el último experimento completo de MODEL_KEY.
DATASET_VERSION = "dfire_seed42_val10_v1"
IMAGE_SIZE = 640
BATCH_SIZE = 16
OPERATING_CONFIDENCE = 0.25
MATCH_IOU = 0.50
NMS_IOU = 0.70
ERROR_CHUNK_SIZE = 4
ERROR_RAM_LIMIT_GIB = 6.0
SEED = 42
ALLOW_VALIDATION_RERUN = False
ALLOW_TEST_RERUN = False
USE_FAST_LOCAL_DATASET = True
FAST_DATA_ROOT = None
STAGING_WORKERS = 8


## 2. Elegir el experimento y comprobaciones previas

In [2]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display, Image as DisplayImage
from ultralytics import YOLO

roots = [Path(os.environ["TFM_PROJECT_ROOT"])] if os.environ.get("TFM_PROJECT_ROOT") else []
roots += [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in roots if (p / "tfm_pipeline.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abrir el notebook dentro de la carpeta del proyecto o definir TFM_PROJECT_ROOT.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tfm_pipeline as pipeline
from tfm_evaluation import (
    model_is_end_to_end, run_error_analysis, save_error_gallery,
    split_artifact_paths, write_summary_markdown,
)

PROJECT_ROOT = pipeline.project_root()
if EVALUATION_SPLIT not in {"val", "test"}:
    raise ValueError("EVALUATION_SPLIT debe ser 'val' o 'test'.")
contract = pipeline.validate_prepared_dataset(PROJECT_ROOT, DATASET_VERSION)
manifest = pipeline.rebased_manifest(contract)
experiment = pipeline.resolve_experiment(
    experiment_id=EXPERIMENT_ID, model_key=None if EXPERIMENT_ID else MODEL_KEY,
    root=PROJECT_ROOT,
)
best_model_path = Path(experiment["best_model"])
if not best_model_path.exists():
    raise FileNotFoundError(best_model_path)
evaluation_root = Path(experiment["experiment_root"]) / "evaluation" / EVALUATION_SPLIT
summary_path = evaluation_root / "evaluation_summary.json"
metrics_path = evaluation_root / "metrics.csv"

display(pd.Series({
    "experiment_id": experiment["experiment_id"],
    "model_key": experiment["model_key"],
    "best_model": str(best_model_path),
    "dataset_version": DATASET_VERSION,
    "evaluation_split": EVALUATION_SPLIT,
    "images": int((manifest["split"] == EVALUATION_SPLIT).sum()),
    "negative_images": int(((manifest["split"] == EVALUATION_SPLIT) & (manifest["box_count"] == 0)).sum()),
}, name="valor").to_frame())


,valor
experiment_id,yolo26s_dfire_seed42_20260830T225658Z
model_key,yolo26s
best_model,/workspace/TFM/artifacts/experiments/yolo26s_d...
dataset_version,dfire_seed42_val10_v1
evaluation_split,val
images,1721
negative_images,783


## 3. Métricas estándar

In [3]:
evaluation_metrics = None
evaluation_model = None
if RUN_STANDARD_EVALUATION:
    allow_rerun = ALLOW_VALIDATION_RERUN if EVALUATION_SPLIT == "val" else ALLOW_TEST_RERUN
    if summary_path.exists() and not allow_rerun:
        raise FileExistsError(
            f"{EVALUATION_SPLIT} ya fue evaluado para este experimento: {summary_path}. "
            "Carga el resultado guardado o activa conscientemente el permiso de repetición."
        )
    evaluation_root.mkdir(parents=True, exist_ok=True)
    staging_base = (
        Path(FAST_DATA_ROOT) if FAST_DATA_ROOT
        else pipeline.default_fast_data_root(PROJECT_ROOT)
        if USE_FAST_LOCAL_DATASET
        else PROJECT_ROOT / "artifacts" / "runtime_datasets"
    )
    staged_dataset = pipeline.stage_prepared_dataset(
        contract, fast_base=staging_base, workers=STAGING_WORKERS
    )
    runtime_yaml = pipeline.write_runtime_data_yaml(
        contract, evaluation_root / "data_runtime.yaml", staged_dataset
    )
    evaluation_model = YOLO(str(best_model_path))
    pipeline.validate_class_mapping(evaluation_model.names)
    device = 0 if torch.cuda.is_available() else "cpu"
    evaluation_metrics = evaluation_model.val(
        data=str(runtime_yaml), split=EVALUATION_SPLIT, imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE, device=device, plots=True,
        project=str(evaluation_root), name="ultralytics", exist_ok=False,
        verbose=True,
    )
else:
    print(f"No se ha ejecutado inferencia sobre {EVALUATION_SPLIT}.")


No se ha ejecutado inferencia sobre val.


In [4]:
def metric_array(metric, name):
    value = getattr(metric, name, None)
    if value is None:
        return np.array([], dtype=float)
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=float).reshape(-1)

if evaluation_metrics is not None:
    box = evaluation_metrics.box
    precision, recall = metric_array(box, "p"), metric_array(box, "r")
    ap50, ap = metric_array(box, "ap50"), metric_array(box, "ap")
    rows = [{
        "scope": "all", "precision": float(precision.mean()),
        "recall": float(recall.mean()), "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
    }]
    for class_id, class_name in pipeline.CLASS_NAMES.items():
        rows.append({
            "scope": class_name, "precision": float(precision[class_id]),
            "recall": float(recall[class_id]), "mAP50": float(ap50[class_id]),
            "mAP50_95": float(ap[class_id]),
        })
    summary = {
        "experiment_id": experiment["experiment_id"],
        "model_key": experiment["model_key"],
        "model_path": str(best_model_path),
        "dataset_version": DATASET_VERSION,
        "split": EVALUATION_SPLIT,
        "end_to_end": model_is_end_to_end(evaluation_model),
        "metrics": rows,
        "speed_ms_per_image": {key: float(value) for key, value in evaluation_metrics.speed.items()},
        "evaluation_run_dir": str(Path(evaluation_metrics.save_dir)),
    }
    pipeline.write_json_atomic(summary_path, summary)
    pd.DataFrame(rows).to_csv(metrics_path, index=False)
else:
    candidate = experiment.get("legacy_evaluation_summary") if EVALUATION_SPLIT == "test" else None
    load_path = summary_path if summary_path.exists() else Path(candidate) if candidate else None
    summary = pipeline.read_json(load_path) if load_path and load_path.exists() else None

if summary:
    display(pd.DataFrame(summary["metrics"]).style.format({
        "precision": "{:.4f}", "recall": "{:.4f}",
        "mAP50": "{:.4f}", "mAP50_95": "{:.4f}",
    }))
else:
    display(Markdown(f"**Este experimento todavía no tiene evaluación estándar guardada en `{EVALUATION_SPLIT}`.**"))


,scope,precision,recall,mAP50,mAP50_95
0,all,0.7812,0.7279,0.7928,0.4611
1,smoke,0.8399,0.8073,0.8599,0.5290
2,fire,0.7225,0.6485,0.7257,0.3931


## 4. Análisis de errores

In [5]:
error_table = error_summary = error_output = None
if RUN_ERROR_ANALYSIS:
    staging_base = (
        Path(FAST_DATA_ROOT) if FAST_DATA_ROOT
        else pipeline.default_fast_data_root(PROJECT_ROOT)
        if USE_FAST_LOCAL_DATASET
        else PROJECT_ROOT / "artifacts" / "runtime_datasets"
    )
    staged_dataset = pipeline.stage_prepared_dataset(
        contract, fast_base=staging_base, workers=STAGING_WORKERS
    )
    manifest = pipeline.rebased_manifest(contract, staged_dataset)
    if evaluation_model is None:
        evaluation_model = YOLO(str(best_model_path))
        pipeline.validate_class_mapping(evaluation_model.names)
    error_output, error_table, error_summary = run_error_analysis(
        evaluation_model, manifest, evaluation_root / "error_analysis",
        model_path=best_model_path, manifest_path=contract["manifest_path"],
        conf=OPERATING_CONFIDENCE, match_iou=MATCH_IOU, nms_iou=NMS_IOU,
        imgsz=IMAGE_SIZE, chunk_size=ERROR_CHUNK_SIZE,
        device=0 if torch.cuda.is_available() else "cpu",
        seed=SEED, ram_limit_gib=ERROR_RAM_LIMIT_GIB,
        split=EVALUATION_SPLIT,
        expected_images=int((manifest["split"] == EVALUATION_SPLIT).sum()),
        expected_negatives=int(((manifest["split"] == EVALUATION_SPLIT) & (manifest["box_count"] == 0)).sum()),
    )
    write_summary_markdown(error_output, error_summary)
    error_paths = split_artifact_paths(error_output, EVALUATION_SPLIT)
    for mode, filename in (
        ("hardest", "hardest_examples.png"),
        ("negative_alarms", "negative_false_alarms.png"),
        ("random", "qualitative_predictions.png"),
    ):
        save_error_gallery(
            error_table, error_paths["predictions"],
            error_output / filename, count=12, seed=SEED, mode=mode,
        )
else:
    candidates = sorted(
        (evaluation_root / "error_analysis").glob(
            f"*/{EVALUATION_SPLIT}_error_summary.json"
        ), reverse=True,
    )
    if candidates:
        error_summary = pipeline.read_json(candidates[0])
        error_output = candidates[0].parent
    elif EVALUATION_SPLIT == "test":
        legacy_error = experiment.get("legacy_error_summary")
        if legacy_error and Path(legacy_error).exists():
            error_summary = pipeline.read_json(Path(legacy_error))

if error_summary:
    display(pd.DataFrame(error_summary["box_metrics"]))
    display(pd.DataFrame(error_summary["negative_image_alarms"]))
    print(
        "Umbral operativo:", error_summary["protocol"]["confidence"],
        "· partición:", error_summary["split"],
        "· negativas:", error_summary["negative_images"],
    )
else:
    print("No hay análisis operativo guardado para este experimento.")


,scope,tp,fp,fn,precision,recall,f1
0,smoke,790,193,170,0.803662,0.822917,0.813176
1,fire,791,387,364,0.671477,0.684848,0.678097
2,all_micro,1581,580,534,0.731606,0.747518,0.739476


,scope,negative_images,negative_images_with_alarm,negative_image_false_alarm_rate
0,any,783,7,0.008940
1,smoke,783,5,0.006386
2,fire,783,2,0.002554
3,both,783,0,0.000000


Umbral operativo: 0.25 · partición: val · negativas: 783


## 5. Comparación de los modelos evaluados

Reúne todos los modelos con evaluación guardada en validación. Con los modelos
de la comparación inicial (YOLOv8n, YOLOv8s, YOLO26n y YOLO26s a 640 px)
reproduce las Tablas 1 y 2 de la memoria. Las métricas operativas usan una
confianza de 0,25 y un IoU de emparejamiento de 0,50.

In [6]:
comparison_rows = []
operating_rows = []
experiments = pipeline.list_experiments(PROJECT_ROOT)
for row in experiments.to_dict("records"):
    root = Path(row["experiment_root"])
    possible = root / "evaluation" / EVALUATION_SPLIT / "evaluation_summary.json"
    if EVALUATION_SPLIT == "test" and not possible.exists() and row.get("legacy_evaluation_summary"):
        possible = Path(row["legacy_evaluation_summary"])
    if not possible.exists():
        continue
    saved = pipeline.read_json(possible)
    for metric in saved["metrics"]:
        comparison_rows.append({
            "experiment_id": row["experiment_id"],
            "model_key": row["model_key"],
            **metric,
        })
    error_files = sorted(
        (root / "evaluation" / EVALUATION_SPLIT / "error_analysis").glob(
            f"*/{EVALUATION_SPLIT}_error_summary.json"
        ), reverse=True,
    )
    if error_files:
        errors = pipeline.read_json(error_files[0])
        micro = next(item for item in errors["box_metrics"] if item["scope"] == "all_micro")
        alarms = next(item for item in errors["negative_image_alarms"] if item["scope"] == "any")
        operating_rows.append({
            "experiment_id": row["experiment_id"],
            "precision_micro": micro["precision"],
            "recall_micro": micro["recall"],
            "f1_micro": micro["f1"],
            "neg_con_alarma": f"{alarms['negative_images_with_alarm']}/{alarms['negative_images']}",
        })

comparison = pd.DataFrame(comparison_rows)
if not comparison.empty:
    overall = comparison[comparison["scope"] == "all"][["experiment_id", "model_key", "mAP50", "mAP50_95"]]
    if operating_rows:
        overall = overall.merge(pd.DataFrame(operating_rows), on="experiment_id", how="left")
    display(Markdown("**Comparación global** (Tabla 1 de la memoria)"))
    display(overall.sort_values("mAP50_95", ascending=False).style.format({
        "mAP50": "{:.2%}", "mAP50_95": "{:.2%}", "precision_micro": "{:.2%}",
        "recall_micro": "{:.2%}", "f1_micro": "{:.2%}",
    }, na_rep="—").hide(axis="index"))
    by_class = comparison[comparison["scope"].isin(["smoke", "fire"])].pivot_table(
        index=["experiment_id", "model_key"], columns="scope", values="mAP50_95"
    ).reset_index().rename(columns={"smoke": "mAP50_95_humo", "fire": "mAP50_95_fuego"})
    display(Markdown("**mAP50-95 por clase** (Tabla 2 de la memoria)"))
    display(by_class[["experiment_id", "model_key", "mAP50_95_humo", "mAP50_95_fuego"]]
            .sort_values("mAP50_95_humo", ascending=False)
            .style.format({"mAP50_95_humo": "{:.2%}", "mAP50_95_fuego": "{:.2%}"})
            .hide(axis="index"))
else:
    print("Aún no hay modelos evaluados para comparar.")

**Comparación global** (Tabla 1 de la memoria)

experiment_id,model_key,mAP50,mAP50_95,precision_micro,recall_micro,f1_micro,neg_con_alarma
legacy_yolov8s_baseline,yolov8s,79.44%,46.42%,70.77%,77.73%,74.09%,18/783
yolo26s_dfire_seed42_20260830T225658Z,yolo26s,79.28%,46.11%,73.16%,74.75%,73.95%,7/783
yolo26n_dfire_seed42_20260831T074308Z,yolo26n,76.89%,44.87%,72.12%,72.06%,72.09%,6/783
yolov8n_dfire_seed42_20260830T174159Z,yolov8n,78.05%,44.63%,68.91%,75.56%,72.08%,19/783


**mAP50-95 por clase** (Tabla 2 de la memoria)

experiment_id,model_key,mAP50_95_humo,mAP50_95_fuego
legacy_yolov8s_baseline,yolov8s,53.65%,39.19%
yolo26s_dfire_seed42_20260830T225658Z,yolo26s,52.90%,39.31%
yolo26n_dfire_seed42_20260831T074308Z,yolo26n,52.57%,37.18%
yolov8n_dfire_seed42_20260830T174159Z,yolov8n,51.73%,37.53%


## Siguiente paso

El notebook [05](05_DFire_barrido_umbrales.ipynb) prueba distintos umbrales de
confianza en tres de estos modelos (YOLOv8s, YOLO26n y YOLO26s) para ver
cuántas falsas alarmas genera cada uno.